**YOUR NAMES HERE**

Spring 2026

CS343: Neural Networks

Project 3: Convolutional Neural Networks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.show()
plt.style.use(['seaborn-v0_8-colorblind', 'seaborn-v0_8-darkgrid'])
plt.rcParams.update({'font.size': 20})

np.set_printoptions(suppress=True, precision=7)

# Automatically reload your external source code
%load_ext autoreload
%autoreload 2

## Goal

**Make sure any debug printouts do not appear if `verbose=False`!**

This week, you will test your CNN on the STL-10 dataset! The last step before you can do this is implementing an optimizer to update your network weights during gradient descent. You will implement a few and compare them.

## Task 5: Implement weight optimizers for gradient descent

To change the weights during training, we need an optimization algorithm to have our loss decrease over epochs as we learn the structure of the input patterns. Until now, we used **Stochastic gradient descent (SGD)**, which is the simplest algorithm. You will implement 4 popular algorithms (*3 this week, 1 more next week*):

1. `SGD` (stochastic gradient descent)
2. `SGD_Momentum` (stochastic gradient descent with momentum)
3. `Adam` (Adaptive Moment Estimation)

Implement each of these according to the update equations (the `update_weights()` in each subclass in `optimizer.py`). Let's use $w_t$ in the math below to represent the weights in a layer at time step $t$, $dw$ to represent the gradient of the weights in a layer, and $\eta$ represent the learning rate. We use vectorized notation below (update applies to all weights element-wise). Then:

**SGD**: 

$w_{t} = w_{t-1} - \eta \times dw$

**SGD (momentum)**:

$v_{t} = m \times v_{t-1} - \eta \times dw$

$w_{t} = w_{t-1} + v_t$

where $v_t$ is called the `velocity` at time $t$. At the first time step (0), velocity should be set to all zeros and have the same shape as $w$. $m$ is a constant that determines how much of the gradient obtained on the previous time step should factor into the weight update for the current time step.


**Adam**:

$v_{t} = \beta_1 \times v_{t-1} + (1 - \beta_1)\times dw$

$p_{t} = \beta_2 \times p_{t-1} + (1 - \beta_2)\times dw^2$

$vc = v_{t} / \left (1-(\beta_1^t) \right )$

$pc = p_{t} / \left (1-(\beta_2^t) \right )$

$w_{t} = w_{t-1} - \left ( \eta \times vc \right ) / \left ( \sqrt(pc) + \epsilon \right ) $


Like SGD (momentum), Adam records momentum terms $v$ and $p$. At time step 0, you should initialize them to zeros in an array equal in size to the weights. $vc$ and $pc$ are variables computed on each time step. The remaining quantities are constants. Note that $t$ keeps track of the integer time step, and needs to be incremented on each update. 


In [ ]:
from optimizer import *

####  Test SGD

In [ ]:
rng = np.random.default_rng(0)

wts = np.arange(-3, 3, dtype=np.float64)
d_wts = rng.standard_normal(len(wts))

optimizer = SGD()
optimizer.prepare(wts, d_wts)

new_wts_1 = optimizer.update_weights()
new_wts_2 = optimizer.update_weights()

print(f'SGD: Wts after 1 iter {new_wts_1}')
print(f'SGD: Wts after 2 iter {new_wts_2}')

Output should be:

    SGD: Wts after 1 iter [-3.012573  -1.9867895 -1.0640423 -0.01049    1.0535669  1.9638405]
    SGD: Wts after 2 iter [-3.025146  -1.973579  -1.1280845 -0.02098    1.1071339  1.927681 ]

####  Test SGD_Momentum

In [ ]:
rng = np.random.default_rng(0)

wts = rng.standard_normal((3, 4))
d_wts = rng.standard_normal((3, 4))

optimizer = SGD_Momentum(lr=0.1, m=0.6)
optimizer.prepare(wts, d_wts)

new_wts_1 = optimizer.update_weights()
new_wts_2 = optimizer.update_weights()

print(f'SGD M: Wts after 1 iter\n{new_wts_1}')
print(f'SGD M: Wts after 2 iter\n{new_wts_2}')

Output should be:

    SGD M: Wts after 1 iter
    [[ 0.3582333 -0.1102257  0.7650137  0.1781269]
    [-0.4812435  0.3932251  1.262837   0.8428296]
    [-0.6908818 -1.4020678 -0.556755   0.006175 ]]
    SGD M: Wts after 2 iter
    [[ 0.7302382 -0.075219   0.9643595  0.2952896]
    [-0.394162   0.4438331  1.1969761  0.6760275]
    [-0.6703162 -1.620702  -0.4503238 -0.0500666]]

####  Test Adam

In [ ]:
rng = np.random.default_rng(0)

wts = rng.standard_normal((3, 4))
d_wts = rng.standard_normal((3, 4))

optimizer = Adam(lr=0.1)
optimizer.prepare(wts, d_wts)

new_wts_1 = optimizer.update_weights()
new_wts_2 = optimizer.update_weights()
new_wts_3 = optimizer.update_weights()

print(f'Adam: Wts after 1 iter\n{new_wts_1}')
print(f'Adam: Wts after 2 iter\n{new_wts_2}')
print(f'Adam: Wts after 3 iter\n{new_wts_3}')

Output should be:

    Adam: Wts after 1 iter
    [[ 0.2257302 -0.0321049  0.7404226  0.2049001]
    [-0.4356694  0.4615951  1.204      0.847081 ]
    [-0.6037352 -1.3654215 -0.5232745 -0.058674 ]]
    Adam: Wts after 2 iter
    [[ 0.3257302  0.0678951  0.8404226  0.3049001]
    [-0.3356694  0.561595   1.104      0.747081 ]
    [-0.5037353 -1.4654215 -0.4232745 -0.158674 ]]
    Adam: Wts after 3 iter
    [[ 0.4257302  0.1678951  0.9404226  0.4049001]
    [-0.2356694  0.661595   1.0040001  0.647081 ]
    [-0.4037353 -1.5654215 -0.3232745 -0.258674 ]] 

## Task 6: Write network training methods

Implement methods in `network.py` to actually train the network, using all the building blocks that you have created. The methods to implement are:

- `predict`
- `fit`

## Task 7: Overfitting a convolutional neural network

Usually we try to prevent overfitting, but we can use it as a valuable debugging tool to test out a complex backprop-based neural network. Assuming everything is working, it is almost always the case that we should be able to overfit a tiny dataset with a huge model with tons of parameters (i.e. your CNN). You will use this strategy to verify that your network is working.

Let's use a small amount of real data from STL-10 to perform the overfitting test.

### 7a. Move `load_stl10_dataset` and `preprocess_data.py` from the MLP project

Make the one change to `preprocess_data.py`: In `preprocess_stl`, Re-arrange dimensions of `imgs` so that when it is returned, `shape=(Num imgs, RGB color chans, height, width)` (No longer flatten non-batch dimensions). *Note: You SHOULD still perform the other preprocessing steps (e.g. standardize).*

In [ ]:
import load_stl10_dataset
import preprocess_data
from network import ConvNet4

### 7b. Load in STL-10

If you don't want to wait for STL-10 to download from the internet and resize, copy over your data and numpy folders from your MLP project.

**Note:** The different train/test split here won't work if you hard coded the proportions in your `create_splits` implementation! *This isn't catastrophic, it just means that it will take longer to compute accuracy on the validation set.*

In [ ]:
# Download the STL-10 dataset from the internet, convert it to Numpy ndarray, resize to 16x16
# cache it locally on your computer for faster loading next time.
load_stl10_dataset.purge_cached_dataset()
# preprocess and create splits
x_train, y_train, x_test, y_test, x_val, y_val, x_dev, y_dev = preprocess_data.load_stl10(
    n_train_samps=4578, n_test_samps=400, n_valid_samps=2, n_dev_samps=20)

print ('Train data shape: ', x_train.shape)
print ('Train labels shape: ', y_train.shape)
print ('Test data shape: ', x_test.shape)
print ('Test labels shape: ', y_test.shape)
print ('Validation data shape: ', x_val.shape)
print ('Validation labels shape: ', y_val.shape)
print ('dev data shape: ', x_dev.shape)
print ('dev labels shape: ', y_dev.shape)

### 7c. Train and overfit the network on a small STL-10 sample with each optimizer

#### Goal

If your network works, you should see a drop in loss over epochs to 0 from the initial value of ~2.3.

#### Todo

In 3 separate cells below

1. Create 3 different `ConvNet4` networks.
2. Compile each with a different optimizer — SGD, SGD-M, and Adam.
3. Train each on the **dev** set and validate on the tiny validation set. *Since we are focused on overfitting in this task, we are not paying attention to the validation loss/acc.*

You will be making plots demonstrating the overfitting for each optimizer below. **You should train the nets with the same number of epochs such that at least 2/3 of them clearly show training loss convergence to a small value; one optimizer may not converge yet, and that's ok**. Cut off the simulations based on the 2/3 that do converge.

#### Guidelines

- To set up a fair comparison across optimizers, you should train your networks with the same hyperparameters each time (*e.g. don't use a different learning rate for different optimizers*).
- Weight scales and learning rates of `5e-3` should work well. You can specify the learning rate with the `compile` method. For example: `net.compile('adam', lr=5e-3)`.
- Try the optimizer that will train the network the fastest first.
- The default hyperparameters should work well (aside from the aforementioned changes to learning rate and weight scale). Keep in mind that you **will** need to change the batch size (*what is your N here?*). I recommend making the batch size as large as possible, given the dataset you are working with otherwise it may be tricky to see whether the loss is actually decreasing on average.
- Decreasing `print_every` will make the `fit` function evaluate the training and validation accuracy more often. This is a computationally intensive process, so small values come with an increase in training time. On the other hand, checking the accuracy too infrequently means you won't know whether the network is trending toward overfitting the training data, which is what you're checking for.
- Each training session takes ~10 mins on my laptop.

#### Caveat emptor

Training convolutional networks is notoriously computationally intensive. If you experiment with hyperparameters, each training session may take several hours.

- Use the loss/accuracy print outs to quickly gauge whether your network is training effectively (*i.e. don't wait until the end of training to monitor loss*).
- Monitor print outs and interrupt the Jupyter kernel if things are not trending in the right direction.

Here is an example of helpful printouts during training:

```
Starting to train...
blah iterations. 1 iter/epoch.
Finished Iter 0/blah. Loss: 2.302
  2.59 sec to finish 1 iter. Estimated total time: 5.18 mins.
  Train acc: 0.25, Val acc: 0.5
Finished Iter 10/blah. Loss: some_number
...
```

In [ ]:
# YOUR CODE HERE

In [ ]:
# Adam

# YOUR CODE HERE

In [ ]:
# SGD-M

# YOUR CODE HERE

In [ ]:
# SGD

# YOUR CODE HERE

### 7d. Evaluate and plot the different optimizer results

Make 2 "high quality" plots showing the following

- Plot the training accuracy (y axis) for the three optimizers as a function of `print_every` training iterations (x axis).
- Plot the training loss (y axis) for the three optimizers as a function of training iterations (x axis).

A high quality plot consists of:
- A useful title
- X and Y axis labels
- A legend

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

### 7e. Questions

**Question 4**: Which optimizer works best and why do think it is best?

**Question 5**: What is happening with the training set accuracy and why?


**Answer 4:**

YOUR ANSWER HERE


**Answer 5:**

YOUR ANSWER HERE

## Task 8: Training your convolutional neural network on STL-10

Now you will train your CNN on the STL-10 training set!

### 8a. Load in STL-10

Here is the code to load in the dataset again in case it is not in memory.

In [ ]:
# Download the STL-10 dataset from the internet, convert it to Numpy ndarray, resize to 32x32
# cache it locally on your computer for faster loading next time.
load_stl10_dataset.purge_cached_dataset()
# preprocess and create splits
x_train, y_train, x_test, y_test, x_val, y_val, x_dev, y_dev = preprocess_data.load_stl10(
    n_train_samps=4000, n_test_samps=500, n_valid_samps=499, n_dev_samps=1)

print ('Train data shape: ', x_train.shape)
print ('Train labels shape: ', y_train.shape)
print ('Test data shape: ', x_test.shape)
print ('Test labels shape: ', y_test.shape)
print ('Validation data shape: ', x_val.shape)
print ('Validation labels shape: ', y_val.shape)
print ('dev data shape: ', x_dev.shape)
print ('dev labels shape: ', y_dev.shape)

### 8b. Set up accelerated convolution and max pooling layers

As you may have noticed, we had to train the network on the dev set (N=20) in a reasonable amount of time. The training set has 4000 samples, how will we ever manage to process that amount of data?!

On one hand, this is an unfortunate inevitable reality of working with large ("big") datasets: you can always find a dataset that is too time consuming to process for any computer, despite how fast/many CPU/GPUs it has.

On the other hand, we can do better for this project and STL-10 :) If you were to time (profile) different parts of the training process, you'd notice that largest bottleneck is convolution and max pooling operations (both forward/backward). You implemented those operations intuitively, which does not always yield the best performance. **By swapping out forward/backward convolution and maxpooling for implementations that use different algorithms (im2col, reshaping) that are compiled to optimized machine code tailored for your computer using the JAX library, we will speed up training up by several orders of magnitude**. JAX should automatically "max out" all the CPU cores your computer has available to speed up training.

Follow these steps to substitute in the "accelerated" convolution and max pooling layers.

1. Create a class called `Conv4NetAccel` in `network.py` by copy-pasting the contents of `Conv4Net`.
2. Import `accelerated_layer` at the top and replace the `Conv2D` and `MaxPooling2D` layers with `Conv2DAccel` and `MaxPooling2DAccel`.

### 8c. Training convolutional neural network on STL-10

You are now ready to train on the entire training set.

- Create a `Conv4NetAccel` object with hyperparameters of your choice.
- Your goal is to achieve at least 45% accuracy on the test and/or validation set.

#### Notes

- This should not be overly challenging. If the default hyperparameters are not getting you to 45%, you may have to change several hyperparameters. Try to change as few as possible. I suggest using your intuition about which hyperparameters to change and monitoring loss and accuracy during training to see if the change is effective (*rather than doing a grid search*). 
- Use the best / most efficient optimizer based on your prior analysis.
- It should take roughly 1 sec per training iteration. If that's way off, seek help as something could be wrong with running the accelerated code.

In [ ]:
from network import ConvNet4Accel

In [ ]:
# YOUR CODE HERE

### 8d. Analysis of STL-10 training quality

Use your trained network that achieves 45%+ accuracy on the test set to make "high quality" plots showing the following 

- Plot the accuracy of the training and validation sets as a function of training over the course of training in units of `print_every` training iterations.
- Plot the loss as a function of training iteration.

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

### 8e. Visualize layer weights

Run the following code and submit the inline image of the weight visualization of the 1st layer (convolutional layer) of the network.

**Note:**
- Setting optional parameter to `True` will let you save a .PNG file in your project folder of your weights. I'd suggest setting it to `False` unless look at your weights and they look like they are worth saving. You don't want a training run that produces undesirable weights to overwrite your good looking results!

In [ ]:
def plot_weights(wts, saveFig=True, filename='convWts.png'):
    grid_sz = int(np.sqrt(len(wts)))
    plt.figure(figsize=(5,5))
    for x in range(grid_sz):
        for y in range(grid_sz):
            lin_ind = np.ravel_multi_index((x, y), dims=(grid_sz, grid_sz))
            plt.subplot(grid_sz, grid_sz, lin_ind+1)
            currImg = wts[lin_ind]
            low, high = np.min(currImg), np.max(currImg)
            currImg = 255*(currImg - low) / (high - low)
            currImg = currImg.astype('uint8')
            plt.imshow(currImg)
            plt.gca().axis('off')
    plt.subplots_adjust(wspace=0.01, hspace=0.01)
    if saveFig:
        plt.savefig(filename)
    plt.show()

In [ ]:
# Subsitute your trained network below
# netT is my network's name
# Every weight should not look like RGB noise
plot_weights(netT.layers[0].wts.transpose(0, 2, 3, 1), saveFig=True, filename='filters.png')

### 8f. Questions

**Question 6:** What do the learned filters look like? Does this make sense to you / is this what you expected? In which area of the brain do these filters resemble cell receptive fields?

#### Note

You should not see all RGB "noise" (some is all right though). If you do, and you pass the "overfit" test with the Adam optimizer, you probably need to increase the number of training epochs.

**Answer 6:**

YOUR ANSWER HERE